# 01 · Bronze: ingestão com marca d'água

Este notebook constrói a camada **bronze**. Ele lê os arquivos Parquet brutos do volume `growdev.bronze.landing` e grava por **append** nas tabelas `growdev.bronze.<tabela>`. Em cada linha, adiciona duas colunas de controle:

- `_ingested_at`: quando o dado foi ingerido (marca d'água).
- `_source_file`: de qual arquivo a linha veio.

Com `_source_file`, a bronze ingere **só os arquivos novos**. Se o job rodar de novo e nenhum arquivo tiver chegado, nada é duplicado.

**Parâmetros**
- `catalogo`: catálogo do projeto (padrão `growdev`).
- `tabela`: `todas` ingere as 4 tabelas, ou o nome de uma só (usado no **For each** da aula 16).

In [ ]:
dbutils.widgets.text("catalogo", "growdev", "Catálogo")
dbutils.widgets.text("tabela", "todas", "Tabela (ou 'todas')")

CATALOGO = dbutils.widgets.get("catalogo")
TABELA = dbutils.widgets.get("tabela")
print(f"Catálogo: {CATALOGO} | Tabela: {TABELA}")

In [ ]:
SCHEMA = "bronze"
VOLUME_LANDING = f"/Volumes/{CATALOGO}/{SCHEMA}/landing"

# Dicionário mapeando cada tabela bronze -> caminho do dado bruto na landing
tabelas_bronze = {
    "account_balances": f"{VOLUME_LANDING}/account_balances",
    "accounts": f"{VOLUME_LANDING}/accounts",
    "customers": f"{VOLUME_LANDING}/customers",
    "loans": f"{VOLUME_LANDING}/loans",
}

# Se recebemos uma tabela só, ficamos apenas com ela
if TABELA != "todas":
    tabelas_bronze = {TABELA: tabelas_bronze[TABELA]}

print(f"Volume de origem: {VOLUME_LANDING}")
print("Tabelas a ingerir:", list(tabelas_bronze.keys()))

In [ ]:
from pyspark.sql import functions as F

def ingerir_bronze(nome_tabela, caminho_landing):
    """Lê o Parquet bruto da landing e faz append na bronze, só dos arquivos novos."""
    tabela_destino = f"{CATALOGO}.{SCHEMA}.{nome_tabela}"  # ex.: growdev.bronze.customers

    df_bronze = (
        spark.read.parquet(caminho_landing)                       # 1. lê o dado bruto
        .withColumn("_ingested_at", F.current_timestamp())        # 2. marca d'água da ingestão
        .withColumn("_source_file", F.col("_metadata.file_path"))  # 3. arquivo de origem
    )

    # 4. Se a tabela já existe, descarta os arquivos que já foram ingeridos antes
    if spark.catalog.tableExists(tabela_destino):
        ja_ingeridos = spark.table(tabela_destino).select("_source_file").distinct()
        df_bronze = df_bronze.join(ja_ingeridos, "_source_file", "left_anti")

    novas_linhas = df_bronze.count()
    if novas_linhas == 0:
        return tabela_destino, 0

    df_bronze.write.mode("append").saveAsTable(tabela_destino)  # 5. append na bronze
    return tabela_destino, novas_linhas

In [ ]:
for nome_tabela, caminho in tabelas_bronze.items():
    destino, novas = ingerir_bronze(nome_tabela, caminho)
    print(f"OK: {nome_tabela} -> {destino} ({novas} linhas novas)")

In [ ]:
# 1) Resumo por tabela: total de linhas e quantos arquivos já foram ingeridos
for nome_tabela in tabelas_bronze:
    df = spark.table(f"{CATALOGO}.{SCHEMA}.{nome_tabela}")
    total = df.count()
    arquivos = df.select("_source_file").distinct().count()
    print(f"{nome_tabela}: {total} linhas vindas de {arquivos} arquivo(s)")

# 2) Exemplo com select + filter + groupBy: clientes por estado na bronze
if "customers" in tabelas_bronze:
    display(
        spark.table(f"{CATALOGO}.{SCHEMA}.customers")
        .select("customer_id", "full_name", "state", "_ingested_at")
        .groupBy("state")
        .count()
        .orderBy(F.desc("count"))
        .limit(10)
    )